# Annotation of MS1 features to authentic cpd libs and SRM1950 libs

This uses authentic compound library in Li lab, and SRM1950 annotation. 
The SRM1950 is a human plasma sample that was included in the experiment as a control. 

Exports full JSON (redudnat) and concise tsv files. 

Not using MS/MS here. Not using CSM as the SRM1950 annotation carries much of CSM, to simplify the process. 

To check if LC-MS methods match to those of authentic lib.

In [7]:
import os
import sys
import json
import pandas as pd

from asari.tools import match_features as mf
from asari.tools.file_io import read_features_from_asari_table


from datetime import date


In [19]:
def get_concise_annotation(feature_id, matched_list, feature_dict, lib_dict):
    """Remove duplicate annotations for the same feature, and return the concise list of fields."""
    feature = feature_dict[feature_id]
    result = {
        "id": feature_id,
        "mz": feature["mz"],
        "rtime": feature["rtime"],
    }
    lib_id, csm_id = None, None
    for x in matched_list:
        if lib_id is None and x.startswith("v2r2024"):
            lib_id = x
        elif csm_id is None and x.startswith("r1_"):
            csm_id = x
        if lib_id and csm_id:
            break
    # for authentic cpd lib and srm/csm matches, keep one each
    if lib_id:
        lib_info = lib_dict[lib_id]
        result.update(
            {
                "lib_id": lib_id,
                "lib_name": lib_info["name"],
                "lib_mz": lib_info["mz"],
                "lib_rtime": lib_info["rtime"],
                "lib_identifier": lib_info["identifier"],
                "lib_ion": lib_info["ion"],
                "lib_isotope": lib_info["isotope"],
            }
        )
    else:
        result.update(
            {
                "lib_id": "",
                "lib_name": "",
                "lib_mz": "",
                "lib_rtime": "",
                "lib_identifier": "",
                "lib_ion": "",
                "lib_isotope": "",
            }
        )
    if csm_id:
        result.update(
            {
                "CSMF_ID": csm_id,
                "CSM_ion": lib_dict[csm_id]["ion_csm"],
                "CSM_top_recommendation_name": lib_dict[csm_id][
                    "top_recommendation_name"
                ],
                "CSM_top_recommendation_score": lib_dict[csm_id][
                    "top_recommendation_score"
                ],
                "CSM_HMDB": lib_dict[csm_id]["HMDB"],
                "CSM_mz": lib_dict[csm_id]["mz"],
                "CSM_rtime": lib_dict[csm_id]["rtime"],
            }
        )
    else:
        result.update(
            {
                "CSMF_ID": "",
                "CSM_ion": "",
                "CSM_top_recommendation_name": "",
                "CSM_top_recommendation_score": "",
                "CSM_HMDB": "",
                "CSM_mz": "",
                "CSM_rtime": "",
            }
        )
    return result

## Library Setup

In [3]:
# Authentic compound libraries; same as in Li lab GitHub repo
indir = "/Users/chongj/Desktop/Li_Lab/annotation_sources/"

authLib_hilicpos = json.load(
    open(f"{indir}/authentic_cpd_library/v3_authlib2024_hilicpos.json")
)
authLib_hilicneg = json.load(
    open(f"{indir}/authentic_cpd_library/v3_authlib2024_hilicneg.json")
)
authLib_rppos = json.load(
    open(f"{indir}/authentic_cpd_library/v3_authlib2024_rppos.json")
)
authLib_rpneg = json.load(
    open(f"{indir}/authentic_cpd_library/v3_authlib2024_rpneg.json")
)
len(authLib_hilicneg), len(authLib_rppos), len(authLib_hilicpos), len(authLib_rpneg)

(748, 950, 760, 1383)

In [4]:
# SRM libs
srmLib_hilicpos = json.load(
    open(f"{indir}/Li_lab_srm1950lib_v1/hilicpos_srm1950lib_v1.json")
)
srmLib_hilicneg = json.load(
    open(f"{indir}/Li_lab_srm1950lib_v1/hilicneg_srm1950lib_v1.json")
)
srmLib_rppos = json.load(open(f"{indir}/Li_lab_srm1950lib_v1/rppos_srm1950lib_v1.json"))
srmLib_rpneg = json.load(open(f"{indir}/Li_lab_srm1950lib_v1/rpneg_srm1950lib_v1.json"))
len(srmLib_hilicneg), len(srmLib_rppos), len(srmLib_hilicpos), len(srmLib_rpneg)

(4518, 11771, 4035, 5425)

In [9]:
# project data

heu_dir = "/Volumes/T7/Li Lab/Projects/HEU/OLD/2025-12-14_HEU_Metabolomics_V3/heu_data/12_03_2025_feature_tables/"

feature_tables = [
    (f"{heu_dir}HILIC_pos_final_table.tsv", "hilicpos"),
    (f"{heu_dir}HILIC_neg_final_table.tsv", "hilicneg"),
    (f"{heu_dir}RP_pos_final_table.tsv", "rppos"),
    (f"{heu_dir}RP_neg_final_table.tsv", "rpneg"),
]

## Running the annotation

In [21]:
output_dir = "/Users/chongj/Desktop/Li_Lab/Projects/HEU/2026-05-21_MS1/"
today = str(date.today())

this_output_dir = f"{output_dir}{today}_csm_srm_annotations/"

os.makedirs(this_output_dir, exist_ok=True)

for feature_table, mode in feature_tables:
    num_samples, list_features = read_features_from_asari_table(
        open(feature_table).read()
    )
    feature_dict = {feature["id"]: feature for feature in list_features}

    if mode == "hilicpos":
        cpdLib = authLib_hilicpos + srmLib_hilicpos
    elif mode == "hilicneg":
        cpdLib = authLib_hilicneg + srmLib_hilicneg
    elif mode == "rppos":
        cpdLib = authLib_rppos + srmLib_rppos
    elif mode == "rpneg":
        cpdLib = authLib_rpneg + srmLib_rpneg
    else:
        raise ValueError(f"Unknown mode: {mode}")
    dict_lib = {entry["id"]: entry for entry in cpdLib}

    matched = mf.list_match_lcms_features(
        list_features, cpdLib, mz_ppm=5, rt_tolerance=30
    )
    header = [
        "id",
        "mz",
        "rtime",
        "lib_id",
        "lib_name",
        "lib_mz",
        "lib_rtime",
        "lib_identifier",
        "lib_ion",
        "lib_isotope",
        "CSMF_ID",
        "CSM_ion",
        "CSM_top_recommendation_name",
        "CSM_top_recommendation_score",
        "CSM_HMDB",
        "CSM_mz",
        "CSM_rtime",
    ]
    rows = ["\t".join(header)]
    for feature_id, matched_list in matched.items():
        concise_info = get_concise_annotation(
            feature_id, matched_list, feature_dict, dict_lib
        )
        rows.append("\t".join([str(concise_info[h]) for h in header]))

    con_output_file = f"{this_output_dir}{today}_{mode}_annotation_Conciseoutput.tsv"

    with open(con_output_file, "w") as f:
        f.write("\n".join(rows) + "\n")

    json_output = []
    for feature_id, matched_list in matched.items():
        json_output.append(
            {
                feature_id: {
                    "Feature": feature_dict[feature_id],
                    "Matched_libs": [dict_lib[lib_id] for lib_id in matched_list],
                }
            }
        )

    json_output_file = f"{this_output_dir}{today}_{mode}_annotation_Fulloutput.json"
    with open(json_output_file, "w") as f:
        json.dump(json_output, f, indent=4)

table header looks like: 
   ['id_number', 'mz', 'rtime', 'rtime_left_base', 'rtime_right_base', 'parent_masstrack_id', 'peak_area', 'cSelectivity', 'goodness_fitting', 'snr', 'detection_counts', 'HEU_Batch3_Sample13_HILIC_pos_SZ_07032023_043.mzML', 'Batch2_HEU011_B_HILIC_pos.mzML', 'Batch2_HEU015_M_HILIC_pos.mzML', 'Batch2_HEU026_B_HILIC_pos.mzML', 'Batch2_HEU027_B_HILIC_pos.mzML', 'Batch2_HEU028_B_HILIC_pos.mzML', 'Batch2_HEU029_B_HILIC_pos.mzML', 'Batch2_HEU032_M_HILIC_pos.mzML', 'Batch2_HEU034_B_HILIC_pos.mzML']
Read 7805 feature lines
Of 7805 list1 features, number of uni-direction matched features is 757.
table header looks like: 
   ['id_number', 'mz', 'rtime', 'rtime_left_base', 'rtime_right_base', 'parent_masstrack_id', 'peak_area', 'cSelectivity', 'goodness_fitting', 'snr', 'detection_counts', 'HEU_Batch5_Sample33_HILIC_neg_SZ_07142023_087.mzML', 'Batch2_HEU011_B_HILIC_neg.mzML', 'Batch2_HEU015_M_HILIC_neg.mzML', 'Batch2_HEU026_B_HILIC_neg.mzML', 'Batch2_HEU027_B_HILIC_neg.mz

# Summary

Annotation files were written in current folder for 4 methods. 

The JSON annotation is more complete. 

SRM annotation was used as a proxy for CSM. 